# Structural Connectome QC: Registration First

This notebook is intentionally narrowed to **registration / parcellation-to-DWI fit** only. It removes the broad node-based audit, streamline-assignment audit, visual Plotly panels, and repair execution flow so we can work one step at a time.

Run cells from top to bottom. After each section, stop and review the output before deciding the next action. This notebook does **not** delete, move, quarantine, or regenerate files.

## 1. Setup And QC Output Load

This cell only loads existing QC CSV outputs from disk. If the CSVs are stale or missing, stop here and tell Codex; do not proceed to repair.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path('/home/ec2-user/exp')
QC_DIR = PROJECT_ROOT / 'data/derivatives/qc/sc_matrix_qc'
DERIV_ROOT = PROJECT_ROOT / 'data/derivatives'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 220)

print('Python:', sys.executable)
print('QC_DIR:', QC_DIR)
print('QC outputs exist:', QC_DIR.exists())

def load_csv(name, required=True):
    path = QC_DIR / name
    if not path.exists():
        msg = f'Missing {path}'
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'loaded {name}: {df.shape[0]} rows x {df.shape[1]} cols')
    return df

repair_plan = load_csv('sc_matrix_qc_repair_plan.csv')
registration_targets = load_csv('bbr_registration_repair_targets.csv', required=False)
parcellation_targets = load_csv('post_parcellation_repair_targets.csv', required=False)
source_targets = load_csv('source_failure_repair_targets.csv', required=False)
coverage = load_csv('parc_label_coverage_qc.csv')
matrix_integrity = load_csv('sc_matrix_integrity_subjects.csv')
decisions = load_csv('sc_matrix_qc_decisions.csv', required=False)

## 2. Registration-Lane Summary

Goal: identify subjects whose zero/near-empty SC matrices are most consistent with **AAL_b0 / 5TT / DWI mask registration mismatch**, not downstream streamline-assignment failure.

Interpretation:
- `bbr_registration`: AAL labels exist, but many zero-matrix labels have poor overlap with the 5TT/DWI brain mask.
- `post_parcellation`: AAL labels are missing or implausibly tiny in AAL_b0; this is a parcellation transform/resampling issue, not yet a BBR repair.
- `assignment_or_tracks`: parcellation seems present/aligned enough, but streamlines are not assigned; this is intentionally not the focus of this pass.

In [ ]:
summary = (
    repair_plan.groupby(['repair_lane', 'whole_matrix_qc_status', 'analysis_gate'], dropna=False)
    .size()
    .reset_index(name='n_subjects')
    .sort_values(['repair_lane', 'whole_matrix_qc_status'])
)
display(summary)

by_group = (
    repair_plan.groupby(['group', 'repair_lane'], dropna=False)
    .size()
    .reset_index(name='n_subjects')
    .pivot(index='group', columns='repair_lane', values='n_subjects')
    .fillna(0)
    .astype(int)
)
display(by_group)

print('Registration targets:', len(registration_targets))
print('Parcellation-label targets:', len(parcellation_targets))
print('Source-failure targets:', len(source_targets))

## 3. Registration Failure Evidence Table

This is the first table to review. Subjects with high `n_poor_overlap_zero_nodes`, low `median_mask_overlap_zero_nodes`, very low density, or severe component fragmentation are the strongest registration-first candidates.

In [ ]:
registration_cols = [
    'sid', 'subject_id', 'group', 'whole_matrix_qc_status', 'repair_lane',
    'repair_priority', 'density', 'nonzero_upper_edges', 'n_unexpected_zero_rows',
    'n_poor_overlap_zero_nodes', 'n_label_tiny_or_absent_zero_nodes',
    'n_coverage_ok_zero_nodes', 'median_mask_overlap_zero_nodes',
    'n_components', 'largest_component_frac', 'whole_matrix_reason',
    'recommended_action',
]
registration_cols = [c for c in registration_cols if c in repair_plan.columns]

registration_review = repair_plan[repair_plan['repair_lane'].eq('bbr_registration')].copy()
registration_review = registration_review.sort_values(['repair_priority', 'sid'], ascending=[False, True])

if registration_review.empty:
    print('No bbr_registration subjects in the current repair plan.')
else:
    display(registration_review[registration_cols].head(80))

## 4. Registration Subject Drilldown

Set `QC_SID` to one subject from the table above, run this cell, and inspect the evidence. Leave `QC_SID = None` to automatically choose the highest-priority registration target.

In [ ]:
QC_SID = None  # Example: 'XXX_S_1000_I401540'

if QC_SID is None:
    if registration_review.empty:
        raise ValueError('No registration targets available. Set QC_SID manually or review repair_plan.')
    QC_SID = str(registration_review.iloc[0]['sid'])

print('Selected subject:', QC_SID)

rp = repair_plan[repair_plan['sid'].astype(str).eq(QC_SID)].copy()
if rp.empty:
    raise ValueError(f'{QC_SID} not found in repair plan')
display(rp[[c for c in registration_cols if c in rp.columns]])

cov = coverage[coverage['sid'].astype(str).eq(QC_SID)].copy()
if cov.empty:
    print('No parcellation coverage rows for this subject.')
else:
    cov_summary = (
        cov.groupby('source_status', dropna=False)
        .agg(
            n_labels=('node', 'count'),
            median_label_voxels=('label_voxels', 'median'),
            median_mask_overlap_frac=('mask_overlap_frac', 'median'),
            min_mask_overlap_frac=('mask_overlap_frac', 'min'),
        )
        .reset_index()
        .sort_values('n_labels', ascending=False)
    )
    display(cov_summary)

    problem_cov = cov[cov['source_status'].ne('coverage_ok')].copy()
    keep_cols = [
        'node', 'atlas_label', 'source_status', 'geometry_match',
        'label_voxels', 'mask_overlap_voxels', 'mask_overlap_frac',
        'AAL_b0_path', 'mask_5tt_path', 'mask_read_error',
    ]
    keep_cols = [c for c in keep_cols if c in problem_cov.columns]
    display(problem_cov.sort_values(['source_status', 'mask_overlap_frac', 'node'], na_position='first')[keep_cols].head(80))

primary = matrix_integrity[
    matrix_integrity['sid'].astype(str).eq(QC_SID) & matrix_integrity['weight'].eq('fd_sum')
].copy()
if not primary.empty:
    display(primary[[c for c in [
        'sid', 'weight', 'n_rows', 'n_cols', 'finite_fraction', 'symmetry_max_abs',
        'nonzero_upper_edges', 'density', 'n_unexpected_zero_rows',
        'unexpected_zero_rows', 'matrix_path', 'read_error'
    ] if c in primary.columns]])

## 5. Manual Overlay Commands For The Selected Subject

Use these commands outside the notebook to visually confirm whether `AAL_b0.nii.gz` is aligned with the DWI/5TT brain mask. The notebook prints commands only; it does not launch viewers.

In [ ]:
if 'QC_SID' not in globals():
    raise RuntimeError('Run the subject drilldown cell first so QC_SID is defined.')

cov = coverage[coverage['sid'].astype(str).eq(QC_SID)].copy()
if cov.empty:
    print('No coverage rows, cannot derive overlay paths.')
else:
    first = cov.iloc[0]
    aal_b0 = Path(str(first.get('AAL_b0_path', '')))
    mask_5tt = Path(str(first.get('mask_5tt_path', '')))
    parc_dir = aal_b0.parent if str(aal_b0) else DERIV_ROOT / 'parc' / QC_SID
    fod_dir = mask_5tt.parent if str(mask_5tt) else DERIV_ROOT / 'fod' / QC_SID

    candidates = []
    for d in [parc_dir, fod_dir, DERIV_ROOT / 'dwi_t1_bbr' / QC_SID, DERIV_ROOT / 'eddy' / QC_SID]:
        if d.exists():
            candidates.extend(sorted(d.glob('*b0*.nii*')))
            candidates.extend(sorted(d.glob('*mean*.nii*')))
            candidates.extend(sorted(d.glob('*dwi*.mif')))
    base_img = candidates[0] if candidates else mask_5tt

    print('AAL_b0:', aal_b0)
    print('5TT/DWI mask:', mask_5tt)
    print('Suggested base image:', base_img)
    print('\nCopy/paste one at a time in a terminal:')
    print(f'mrinfo "{base_img}" "{aal_b0}" "{mask_5tt}"')
    print(f'mrview "{base_img}" -overlay.load "{mask_5tt}" -overlay.load "{aal_b0}"')
    print(f'mrstats "{aal_b0}" -mask "{mask_5tt}"')

## 6. Disabled Registration Repair Commands

Do **not** run repairs yet. First, run the cells above, inspect the registration table and overlays, and then decide which repair lane is justified.

In [ ]:
RUN_REPAIR = False

registration_sids = registration_review['sid'].astype(str).tolist() if 'registration_review' in globals() else []
print(f'{len(registration_sids)} registration candidates are available.')
print('RUN_REPAIR =', RUN_REPAIR)
print('\nDisabled command sketch only:')
print('# 1) Rerun/check T1-to-B0 BBR bridge for selected subjects.')
print('# 2) Regenerate AAL_b0 after registration is verified.')
print('# 3) Rerun Step 7 post/connectome only for repaired subjects.')
print('# 4) Rerun this QC notebook and require improved density/overlap before analysis inclusion.')

if registration_sids:
    print('\nFirst 20 registration SIDs:')
    for sid in registration_sids[:20]:
        print(sid)

if RUN_REPAIR:
    raise RuntimeError('Repair execution is intentionally disabled in this registration-first notebook.')

## Stop Here

After you run the registration sections, share the output or the selected subject evidence. We will then decide the next step: BBR repair, parcellation regeneration, assignment diagnostics, or no action.

## 7. Supervisor Raw End-to-End Replay

This scratch-only section tests whether the supervisor/reference pipeline can produce a healthy connectome when started from raw ADNI DWI DICOM and raw T1, instead of trying to repair downstream SC matrices in place.

- Production outputs are not overwritten.
- Outputs go under `data/derivatives/qc/sc_matrix_qc/supervisor_raw_end_to_end_probe/`.
- The current EC2 toolchain is missing `ss3t_csd_beta1` and ANTs `N4BiasFieldCorrection`; the launcher therefore records the run as a near-reference fallback when `--allow-fallbacks` is used, not exact supervisor equivalence.
- If this replay generates a dense matrix with few/no unexpected zero rows, the fix belongs upstream in preprocessing/T1-to-B0/AAL routing. If it fails too, the current source contract needs deeper raw-data or toolchain correction before batch repair.


In [ ]:
# Launch scratch-only supervisor raw replay in a terminal/tmux session.
# This does not overwrite production derivatives.
!SID=XXX_S_1001_I1249292 SELECT_STREAMLINES=1000000 NTHREADS=4 /home/ec2-user/exp/run_sc_supervisor_raw_end_to_end_probe_tmux.sh


In [ ]:
# Monitor commands for a regular terminal, not inside the notebook:
# tmux attach -t sc_supervisor_raw_probe
# tmux attach -t sc_supervisor_raw_probe_monitor
# Or one-shot status:
!timeout 3s /home/ec2-user/exp/watch_sc_supervisor_raw_probe.sh || true
